In [28]:
# import h5py
# from tensorflow import keras
# model = keras.models.load_model('Big_Model.h5')
# model.summary()

In [1]:
# Import AI stuff and load data
import pandas as pd
import numpy as np

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

train_df = pd.read_csv("train_data.csv")
test_df = pd.read_csv("test_data.csv")

(train_df.shape, test_df.shape)

((2109600, 8), (451440, 8))

In [2]:
# Get rid of any useless cols that doesnt help prediction
useless_cols = ['timestamp', 'encounter_id']
train_df = train_df.drop(useless_cols, axis=1)
test_df = test_df.drop(useless_cols, axis=1)

train_df.head()

,heart_rate,systolic_bp,diastolic_bp,respiratory_rate,oxygen_saturation,label
0,88.0,104.0,74.0,18.0,95.0,0
1,88.0,100.0,72.0,18.0,95.0,0
2,86.0,96.0,70.0,17.0,94.0,0
3,89.0,98.0,70.0,18.0,94.0,0
4,91.0,98.0,70.0,NaN,94.0,0


In [3]:
# Get rid of any entries with null vals
train_df = train_df.dropna()
test_df = test_df.dropna()
train_df.shape, test_df.shape


((1906801, 6), (408576, 6))

In [4]:
# Create training and testing sets - also separate features/labels
X_train = train_df.drop(columns = ["label"])
y_train = train_df["label"]

X_test = test_df.drop(columns = ["label"])
y_test = test_df["label"]

In [5]:
from tensorflow.keras.utils import to_categorical

unique_states = sorted(y_train.unique())
num_classes = len(unique_states)

# Create a mapping from np.int to actual int
state_to_index = {state: i for i, state in enumerate(unique_states)}

# Convert our training labels to numerical indices
y_train_indices = y_train.map(state_to_index)

# One-hot encode the training labels
y_train_encoded = to_categorical(y_train_indices, num_classes=num_classes)
print("y_train_encoded shape:", y_train_encoded.shape, 
      "Example one-hot vector:", y_train_encoded[0])


# Repeat for testing data
unique_states = sorted(y_test.unique())
num_classes = len(unique_states)

state_to_index = {state: i for i, state in enumerate(unique_states)}

y_test_indices = y_test.map(state_to_index)

y_test_encoded = to_categorical(y_test_indices, num_classes=num_classes)
print("y_train_encoded shape:", y_test_encoded.shape, 
      "Example one-hot vector:", y_test_encoded[0])

y_train_encoded shape: (1906801, 4) Example one-hot vector: [1. 0. 0. 0.]
y_train_encoded shape: (408576, 4) Example one-hot vector: [1. 0. 0. 0.]


In [6]:
model = Sequential()

# Input layer: our features are the vitals
model.add(Dense(64, activation='relu', input_shape=(X_train.shape[1],)))
model.add(Dropout(0.2))

# Hidden layer 2
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.2))

# Hidden layer 3
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.2))

# Output layer for multi-class
model.add(Dense(num_classes, activation='softmax'))

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

c:\Users\ragha\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,964 (35.02 KB)

 Trainable params: 8,964 (35.02 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
EPOCHS = 3
BATCH_SIZE = 128

history = model.fit(
    X_train,
    y_train_encoded,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.1,  # 90% train, 10% validation
    verbose=1
)

Epoch 1/3
13408/13408 ━━━━━━━━━━━━━━━━━━━━ 44s 3ms/step - accuracy: 0.7676 - loss: 0.6058 - val_accuracy: 0.7984 - val_loss: 0.5581
Epoch 2/3
13408/13408 ━━━━━━━━━━━━━━━━━━━━ 44s 3ms/step - accuracy: 0.7828 - loss: 0.5434 - val_accuracy: 0.7777 - val_loss: 0.6641
Epoch 3/3
13408/13408 ━━━━━━━━━━━━━━━━━━━━ 46s 3ms/step - accuracy: 0.7877 - loss: 0.5259 - val_accuracy: 0.6829 - val_loss: 0.8460


In [8]:
model.evaluate(X_test, y_test_encoded)

12768/12768 ━━━━━━━━━━━━━━━━━━━━ 21s 2ms/step - accuracy: 0.6916 - loss: 0.8385


[0.8385306596755981, 0.6916412115097046]